In [0]:
%sql
SELECT
  current_catalog() AS current_catalog,
  current_schema() AS current_schema;

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.revenue_leakage_bronze
COMMENT 'Raw and incrementally ingested source data';

CREATE SCHEMA IF NOT EXISTS workspace.revenue_leakage_silver
COMMENT 'Cleaned, validated and deduplicated data';

CREATE SCHEMA IF NOT EXISTS workspace.revenue_leakage_gold
COMMENT 'Business-ready fact, dimension and KPI tables';

CREATE SCHEMA IF NOT EXISTS workspace.revenue_leakage_quarantine
COMMENT 'Invalid records rejected by data-quality rules';

CREATE SCHEMA IF NOT EXISTS workspace.revenue_leakage_monitoring
COMMENT 'Pipeline execution and data-quality metrics';

SHOW SCHEMAS IN workspace LIKE 'revenue_leakage*';

In [0]:
%sql
SHOW SCHEMAS IN workspace LIKE 'revenue_leakage*';

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS workspace.revenue_leakage_bronze.landing
COMMENT 'Landing zone for incoming files from simulated source systems';

SHOW VOLUMES IN workspace.revenue_leakage_bronze;

In [0]:
landing_path = "/Volumes/workspace/revenue_leakage_bronze/landing"

source_datasets = [
    "ecommerce/orders",
    "ecommerce/order_items",
    "ecommerce/products",
    "ecommerce/deliveries",
    "payment_system/payments",
    "payment_system/refunds",
    "crm/customers",
    "support/tickets"
]

created_paths = []

for dataset in source_datasets:
    full_path = f"{landing_path}/{dataset}"
    dbutils.fs.mkdirs(full_path)
    created_paths.append((dataset, full_path))

paths_df = spark.createDataFrame(
    created_paths,
    ["source_dataset", "volume_path"]
)

display(paths_df)